In [1]:
import pandas as pd
from src.preprocessing import train_valid_test_split
from src.evaluate import precision_k
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer


In [2]:
df = pd.read_csv('../data/bank-full.csv', sep=';')
train, valid, test = train_valid_test_split(df)
k = int(len(valid) * 0.1)

In [3]:
#baseline 0
baseline_0_precision_k = valid["y"].eq("yes").mean()
print(baseline_0_precision_k)

0.11515777056915365


In [4]:
#baseline 1
X_train = train.drop(columns=['y', 'duration'])
y_train = train["y"]
cat_cols = X_train.select_dtypes(
    include=["object", "category", "string", "bool"]
).columns.tolist()

model = DecisionTreeClassifier(
    criterion="entropy",
    min_samples_leaf=5,
    min_samples_split=5,
    max_depth=5,
    random_state=42,
)
encoder = OneHotEncoder()

cat_pipe = Pipeline(
    [('ohe', OneHotEncoder(handle_unknown='ignore'))]
)

preprocessor = ColumnTransformer(
    [
        ('cat', cat_pipe, cat_cols)
    ]
)

pipe = Pipeline(
    [
        ('preprocessor', preprocessor),
        ("model", model),
    ]
)

In [5]:
X_valid = valid.drop('y', axis=1)
y_valid = valid['y']
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_valid)

In [8]:
X_valid = valid[X_train.columns]
pipe.fit(X_train, y_train)

yes_idx = list(pipe.classes_).index("yes")
y_score = pipe.predict_proba(X_valid)[:, yes_idx]
prec_k = precision_k(valid["y"], y_score, k=k)
print(prec_k)

0.2079646017699115
